# Proxy Clinical: Block 2 pilot launcher (Colab Files, no Drive)

Thin launcher. No training logic lives here; everything runs from the repo.
Runtime: **A100 (high-RAM)**.

**Upload first** (Files panel on the left, upload icon): `proxy-clinical-block2.tar.gz`. It contains the
repo and the pilot JSONL, so it is the only file needed. Optional: add `HF_TOKEN` in the Secrets panel
(only needed for gated bases such as Llama).

`/content` is wiped when the runtime is deleted, not when the session is restarted. So: run cell 1, cell 2,
then **Runtime > Restart session**, then cells 3 and 4. The uploaded tarball and the extracted repo survive
the restart.

**Cell 4 downloads the results itself** the moment the run finishes: first `predictions.jsonl` (the one file
needed for error analysis), then the full run zip (adapter, manifests, eval, determinism; checkpoints excluded).
The browser may ask once to allow multiple downloads from this site; allow it. Cell 5 only re-downloads.
Nothing on this runtime persists, so if the downloads did not land, run cell 5 before the runtime times out.


In [ ]:
# 1. Extract the uploaded tarball into /content/proxy-clinical
import os, subprocess, glob
TARBALLS = sorted(glob.glob('/content/proxy-clinical-block*.tar.gz'))
assert TARBALLS, 'upload proxy-clinical-block2.tar.gz to /content via the Files panel first'
TARBALL = TARBALLS[-1]
REPO = '/content/proxy-clinical'
# Always re-extract so a newly uploaded tarball replaces the old tree (runs/ and the HF cache live outside it).
subprocess.run(['rm', '-rf', REPO], check=True)
subprocess.run(['tar', '-xzf', TARBALL, '-C', '/content'], check=True)
print('repo:', REPO)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
for name in ('corpus.jsonl', 'train.jsonl', 'val.jsonl', 'meta.json'):
    p = f'{REPO}/data/pilot/{name}'
    print(f'{name:14s}', 'ok' if os.path.exists(p) else 'MISSING', os.path.getsize(p) if os.path.exists(p) else '')


In [ ]:
# 2. Pinned environment (replaces Colab's preinstalled stack). Then: Runtime > Restart session, continue at cell 3.
# Colab preinstalls torchvision/torchaudio (built against its own torch) and an old torchao; after torch is replaced
# they fail to load, and transformers/peft surface that as bogus import errors. Nothing here needs them.
%pip uninstall -y -q torchvision torchaudio torchao bitsandbytes
%pip install -q -r /content/proxy-clinical/requirements-colab.txt
import subprocess, sys
out = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True).stdout.strip()
print('torch after install:', out)
if out.endswith('False'):
    # Driver older than the CUDA 13 wheel needs: same torch version, CUDA 12.6 build.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'torch==2.14.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
    print('reinstalled torch (cu126 build)')
chk = subprocess.run([sys.executable, '-c', 'import transformers, trl, peft; from transformers import AutoModelForCausalLM; print("imports ok")'],
                     capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-800:])
print('Now: Runtime > Restart session, then run cells 3, 4, 5.')


In [ ]:
# 3. (after restart) Sanity checks + preflight: import probes, then a one-step CPU training on the tiny stand-in.
# Fails in ~1 minute if the environment is broken, BEFORE the 6 GB model download. Also pulls HF_TOKEN from Secrets.
import os, hashlib
os.chdir('/content/proxy-clinical')
EXPECTED = {'corpus.jsonl': 'fe0dad56f62d7bc5', 'train.jsonl': '2b24a41670e462a3', 'val.jsonl': '43c1f81b2a279cfb'}  # cafebabe pilot, v2 contract
for name, want in EXPECTED.items():
    p = f'data/pilot/{name}'
    got = hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16]
    print(f'{name:14s}', got, os.path.getsize(p), 'bytes', 'ok' if got == want else f'!= expected {want} (wrong tarball or contract?)')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Secrets')
except Exception as e:
    print('HF_TOKEN not set (fine for Qwen2.5, required for Llama):', type(e).__name__)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!bash scripts/preflight.sh

In [ ]:
# 4. Run the pilot: train -> infer -> evaluate -> determinism -> pack -> DOWNLOAD.
# Re-run with the same RUN_ID to resume within this session. The download happens here, not in a later cell,
# because /content is gone the moment the runtime is reclaimed.
import datetime, os, subprocess
os.chdir('/content/proxy-clinical')
RUN_ID = os.environ.get('RUN_ID') or f"pilot-qwen2.5-3b-v2-{datetime.date.today():%Y%m%d}"
OUT = f'/content/runs/{RUN_ID}'
print('run folder:', OUT)
rc = subprocess.run(['bash', 'scripts/run_pilot.sh', 'configs/pilot.yaml', OUT]).returncode

from google.colab import files
preds, zip_path = f'{OUT}/predictions.jsonl', f'{OUT}.zip'
if os.path.exists(preds):
    files.download(preds)                       # small; the file needed for error analysis
if rc == 0 and os.path.exists(zip_path):
    print('zip:', zip_path, round(os.path.getsize(zip_path) / 1e6, 1), 'MB;', open(zip_path + '.sha256').read().strip())
    files.download(zip_path)                    # adapter, manifests, eval, determinism, pip freeze
else:
    # The run did not finish: pack whatever exists so nothing is lost, then re-run this cell to resume.
    subprocess.run(['python', 'scripts/pack_run.py', OUT, '--partial', '--quiet'])
    if os.path.exists(zip_path):
        files.download(zip_path)
    raise SystemExit(f'run_pilot.sh exited {rc}; partial artifacts downloaded, re-run this cell to resume from the last checkpoint')

In [ ]:
# 5. Re-download only (cell 4 already did this). Repacks the run folder and downloads it again.
import os, datetime, subprocess
from google.colab import files
os.chdir('/content/proxy-clinical')
RUN_ID = os.environ.get('RUN_ID') or f"pilot-qwen2.5-3b-v2-{datetime.date.today():%Y%m%d}"
OUT = f'/content/runs/{RUN_ID}'
assert os.path.isdir(OUT), f'{OUT} is gone: the runtime was recycled and the run must be repeated (cells 1, 2, restart, 3, 4)'
subprocess.run(['python', 'scripts/pack_run.py', OUT, '--partial', '--quiet'], check=True)
print(open(f'{OUT}/eval_report.md').read() if os.path.exists(f'{OUT}/eval_report.md') else '(no eval_report.md yet)')
files.download(f'{OUT}/predictions.jsonl')
files.download(f'{OUT}.zip')

The run zip holds `adapter/` (LoRA weights + tokenizer), `run_manifest.json`, `run_lock.json`, `config.yaml`,
`predictions.jsonl` (+ manifest), `eval_report.md`, `eval.json`, `determinism.json`, `pip_freeze.txt`;
`<zip>.sha256` is written next to it. Checkpoints are excluded (only useful for resuming inside the same runtime).

The determinism claim proven above is: same checkpoint, same inputs, same pinned environment, same session,
byte-identical output. It is not a cross-GPU claim.